# FT 1.5B Solo — RACE Direct Answer Fine-Tuning

**Purpose:** Fine-tune Qwen2.5-1.5B-Instruct to answer RACE reading comprehension questions **directly** (no guide, no pipeline). Then evaluate accuracy on N=300 questions at seed=42.

**Why this matters:** RACE is a multi-choice reading comprehension benchmark (options A/B/C/D). The guided pipeline costs 10.5B parameter-passes. This experiment costs 1.5B parameter-passes (one forward pass). If FT 1.5B solo is competitive, the pipeline's architecture claim needs rethinking.

| Condition | Compute | Expected |
|---|---|---|
| Baseline (1.5B×5) | 7.5B pp | ~45% |
| **FT 1.5B Solo (this run)** | **1.5B pp** | **TBD** |
| Guided pipeline | 10.5B pp | TBD |

> Seed=42 · N=300 · RACE (high+middle combined)

In [1]:
# CELL 1 — Install
# !pip install -q transformers accelerate peft datasets trl huggingface_hub
print("Done.")

Done.


In [ ]:
# CELL 2 — Login
from huggingface_hub import login
login("")  # paste your HF token
print("Login done")

Login done


In [3]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 10.0 MB/s eta 0:00:00 0:00:01


In [4]:
# CELL 3 — Imports
import os, json, re, random, time, math
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from tqdm.notebook import tqdm

OUTPUT_DIR = "/content/race_ft1b5_solo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Output: {OUTPUT_DIR}")

GPU: Tesla T4
VRAM: 15.6 GB
Output: /content/race_ft1b5_solo


In [5]:
# CELL 4 — Config
CONFIG = {
    "model_name"        : "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_name"      : "ehovy/race",
    "dataset_config"    : "all",          # includes both 'high' and 'middle'
    "eval_seed"         : 42,
    "eval_n"            : 700,
    "train_split_seed"  : 0,
    "max_train_samples" : 500,
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "learning_rate"     : 2e-4,
    "num_epochs"        : 3,
    "batch_size"        : 2,
    "grad_accum"        : 8,
    "max_seq_length"    : 512,            # RACE articles can be long
    "eval_temperature"  : 0.0,
    "max_new_tokens"    : 16,             # answer is just A/B/C/D or the option text
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"        : 50,
}
print("Config ready. eval_seed=42, eval_n=300.")

Config ready. eval_seed=42, eval_n=300.


In [6]:
# CELL 5 — Load and split RACE
# RACE format: article, question, options (list of 4), answer (A/B/C/D)
print("Loading RACE (all = high + middle)...")
raw_ds = load_dataset(CONFIG["dataset_name"], CONFIG["dataset_config"])

OPTION_LABELS = ["A", "B", "C", "D"]

def make_race_item(item):
    article  = item.get("article", "").strip()
    question = item.get("question", "").strip()
    options  = item.get("options", [])       # list of 4 strings
    answer   = item.get("answer", "").strip() # "A", "B", "C", or "D"

    # Build a formatted options block
    opts_text = "".join(
        f"{label}. {opt}" for label, opt in zip(OPTION_LABELS, options)
    )

    return {
        "article"  : article,
        "question" : question,
        "options"  : options,
        "opts_text": opts_text,
        "answer"   : answer,          # ground truth label: A/B/C/D
    }

# Combine test split for eval (RACE has train/validation/test)
all_test = [make_race_item(x) for x in raw_ds["test"]]
all_train = [make_race_item(x) for x in raw_ds["train"]]

print(f"RACE test  examples : {len(all_test)}")
print(f"RACE train examples : {len(all_train)}")

# Eval split: seed=42, N=300
random.seed(CONFIG["eval_seed"])
eval_data = random.sample(all_test, CONFIG["eval_n"])

# Train split from train set
random.seed(CONFIG["train_split_seed"])
if len(all_train) > CONFIG["max_train_samples"]:
    train_data = random.sample(all_train, CONFIG["max_train_samples"])
else:
    train_data = all_train

print(f"Eval set : {len(eval_data)} questions (seed=42)")
print(f"Train set: {len(train_data)} questions")
print("Sample eval item:")
print(f"  Article (first 100 chars): {eval_data[0]["article"][:100]}")
print(f"  Question: {eval_data[0]["question"]}")
print(f"  Options:{eval_data[0]["opts_text"]}")
print(f"  Answer: {eval_data[0]["answer"]}")

Loading RACE (all = high + middle)...


README.md: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/37.4M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4934 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/87866 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

RACE test  examples : 4934
RACE train examples : 87866
Eval set : 700 questions (seed=42)
Train set: 500 questions
Sample eval item:
  Article (first 100 chars): HOGN KONG--Nine out of 10 Singapore citizens returned dropped wallets with money in them,but in Hone
  Question: It call be learned from the newspaper that Bombay is a city in  _  .
  Options:A. EuropeB. AsiaC. the United StatesD. prefix = st1 /Malaysia
  Answer: B


In [7]:
# CELL 6 — Format training data as SFT prompt
# The model is trained to output only the letter (A/B/C/D).
# We feed the full article + question + options, and supervise on just the label.

SYSTEM_PROMPT = ("""
You are a careful reading comprehension solver.
Read the passage and question carefully, then output ONLY the letter 
(A, B, C, or D) of the correct answer.
Do not explain. Output the single letter only.
"""
)

def format_sft_race(item, tokenizer):
    user_content = (
        f"Passage:\n{item['article']}\n\n"
        f"Question: {item['question']}\n\n"
        f"Options: {item['opts_text']}"
    )

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": item["answer"]},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    return {"text": text}

print("SFT format ready.")
print("Example:")
ex = train_data[0]
print(f"  Answer: {ex["answer"]}")
print(f"  Question: {ex["question"][:80]}...")

SFT format ready.
Example:
  Answer: B
  Question: On the whole, which of the following is the best way to make driving safer?...


In [8]:
# CELL 7 — Load tokenizer
print(f"Loading tokenizer: {CONFIG["model_name"]}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer ready.")

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.


In [9]:
# CELL 8 — Prepare HF Dataset for SFTTrainer
from datasets import Dataset

train_formatted = [format_sft_race(x, tokenizer) for x in train_data]
hf_train = Dataset.from_list(train_formatted)

print(f"Training examples: {len(hf_train)}")
print(f"Sample text (first 300 chars): {hf_train[0]["text"][:300]}")

Training examples: 500
Sample text (first 300 chars): <|im_start|>system

You are a careful reading comprehension solver.
Read the passage and question carefully, then output ONLY the letter 
(A, B, C, or D) of the correct answer.
Do not explain. Output the single letter only.
<|im_end|>
<|im_start|>user
Passage:
Listening to music while you drive can 


In [10]:
# CELL 9 — Load base model
print(f"Loading: {CONFIG["model_name"]}")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.config.use_cache = False
base_model.enable_input_require_grads()
vram = torch.cuda.memory_allocated()/1e9
print(f"VRAM after load: {vram:.2f}GB")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading: Qwen/Qwen2.5-1.5B-Instruct


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM after load: 1.50GB


In [11]:
# CELL 10 — Attach LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print("LoRA attached.")

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
LoRA attached.


In [12]:
# CELL 11 — Training args
print(f"Starting fine-tuning: {CONFIG["num_epochs"]} epochs, {len(hf_train)} examples")

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    dataloader_num_workers=0,
    seed=42,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting fine-tuning: 3 epochs, 500 examples


In [13]:
# CELL 12 — Run training
trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train,
    processing_class=tokenizer,
    args=training_args,
)

t0 = time.time()
trainer.train()
print(f"Training done in {(time.time()-t0)/60:.1f} min.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,2.088348
40,1.813442
60,1.755041
80,1.731030


Training done in 8.9 min.


In [14]:
# CELL 13 — Save fine-tuned model
ft_model_path = f"{OUTPUT_DIR}/ft_model"
trainer.save_model(ft_model_path)
tokenizer.save_pretrained(ft_model_path)
print(f"Model saved to {ft_model_path}")

Model saved to /content/race_ft1b5_solo/ft_model


In [15]:
# CELL 14 — Answer extraction for RACE
# RACE answers are single letters: A, B, C, or D.
# We extract the first valid letter from the model output.

import re

VALID_LABELS = {"A", "B", "C", "D"}

def extract_race_label(text):
    """Extract the answer letter (A/B/C/D) from model output."""
    text = text.strip()

    # 1. First line if it is just a letter
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        first = lines[0].upper().rstrip(".").strip()
        if first in VALID_LABELS:
            return first

    # 2. Explicit patterns: "The answer is A", "Answer: B", etc.
    m = re.search(
        r"(?:answer\s*(?:is|:)|correct\s+answer\s*(?:is|:))\s*([A-D])",
        text, re.IGNORECASE
    )
    if m:
        return m.group(1).upper()

    # 3. Standalone letter anywhere in text (first occurrence)
    m = re.search(r"\b([A-D])\b", text, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    return ""

def normalize_race(s):
    return s.strip().upper()

# Quick sanity tests
_tests = [
    ("A",           "A"),
    ("The answer is B", "B"),
    ("  c  ",        "C"),
    ("D.",           "D"),
    ("Option B is correct.", "B"),
]

ok = all(normalize_race(extract_race_label(t)) == e for t, e in _tests)
print("Extractor:", "ALL PASSED" if ok else "FAIL")

Extractor: ALL PASSED


In [16]:
# CELL 15 — Load fine-tuned model for evaluation
del model, base_model, trainer
torch.cuda.empty_cache()

from peft import PeftModel

eval_base = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
).eval()
ft_model = PeftModel.from_pretrained(eval_base, ft_model_path).eval()
print("Fine-tuned model loaded for evaluation.")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned model loaded for evaluation.
VRAM: 3.22GB


In [17]:
# CELL 16 — Single-pass evaluation function
# FT 1.5B Solo: ONE greedy forward pass (T=0) — not 5 votes.
# Compute = 1.5B × 1 = 1.5B pp

EVAL_SYSTEM = (
    "You are a careful reading comprehension solver.\n"
    "Read the passage and question carefully, then output ONLY the letter "
    "(A, B, C, or D) of the correct answer.\n"
    "Do not explain. Output the single letter only."
)

def run_ft_solo_race(item):
    user_content = (
        f"Passage:\n{item['article']}\n\n"
        f"Question: {item['question']}\n\n"
        f"Options:\n{item['opts_text']}"
    )
    
    messages = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=CONFIG["max_seq_length"]
    )
    
    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,          # greedy — deterministic
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

print("Eval function ready. Single greedy pass — 1.5B pp per question.")

Eval function ready. Single greedy pass — 1.5B pp per question.


In [18]:
# CELL 17 — Verification run (20 questions)
v_correct = 0; v_empty = 0
print("Verification: 20 questions...")
for item in eval_data[:20]:
    raw  = run_ft_solo_race(item)
    pred = normalize_race(extract_race_label(raw))
    gt   = normalize_race(item["answer"])
    if not pred: v_empty += 1
    if pred == gt: v_correct += 1

print(f"Verification: {v_correct}/20 = {v_correct/20*100:.0f}% correct")
print(f"Empty answers: {v_empty}/20")
print("Expected range: 30-70% (model just fine-tuned)")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Verification: 20 questions...
Verification: 15/20 = 75% correct
Empty answers: 1/20
Expected range: 30-70% (model just fine-tuned)


In [19]:
# CELL X — Self-Consistency (5 votes majority)

def run_ft_self_consistency_race(item, votes=5):
    user_content = (
        f"Passage:\n{item['article']}\n\n"
        f"Question: {item['question']}\n\n"
        f"Options:\n{item['opts_text']}"
    )
    
    messages = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": user_content},
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=CONFIG["max_seq_length"]
    )
    
    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    answers = []
    
    for _ in range(votes):
        with torch.no_grad():
            out = ft_model.generate(
                **inputs,
                max_new_tokens=CONFIG["max_new_tokens"],
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        new_toks = out[0][inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(new_toks, skip_special_tokens=True).strip()
        
        label = normalize_race(extract_race_label(text))
        if label in VALID_LABELS:
            answers.append(label)

    if not answers:
        return ""
    
    return Counter(answers).most_common(1)[0][0]

print("Self-consistency function ready (5 votes, temperature=0.7).")

Self-consistency function ready (5 votes, temperature=0.7).


In [21]:
# CELL 19 — Full evaluation (N=300) — Self-Consistency (5 votes)
print(f"Evaluating {CONFIG["eval_n"]} questions — FT Self-Consistency (5 votes)...")
print("Compute: 7.5B pp per question (5 voting passes)")
print("-"*60)

SC_RESULTS_FILE = f"{OUTPUT_DIR}/results_sc.jsonl"
SC_CHECKPOINT   = f"{OUTPUT_DIR}/checkpoint_sc.json"

results_sc = []; start_idx = 0

if os.path.exists(SC_CHECKPOINT):
    with open(SC_CHECKPOINT) as f: ck = json.load(f)
    start_idx = ck.get("last_index", 0)
    if start_idx >= len(eval_data):
        print("Previous SC run completed — starting fresh.")
        start_idx = 0; results_sc = []
    else:
        if os.path.exists(SC_RESULTS_FILE):
            with open(SC_RESULTS_FILE) as f:
                results_sc = [json.loads(l) for l in f if l.strip()]
        print(f"Resuming SC from index {start_idx}")

t0 = time.time()
for idx in tqdm(range(start_idx, len(eval_data)), desc="FT-1.5B-SC(5)"):
    item = eval_data[idx]
    try:
        pred = run_ft_self_consistency_race(item, votes=5)
        gt   = normalize_race(item["answer"])
        results_sc.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"],
            "final_answer": pred,
            "correct": (pred == gt), "empty": (pred == ""),
        })
    except Exception as e:
        results_sc.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"],
            "final_answer": "", "correct": False, "empty": True,
            "error": str(e)
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(SC_RESULTS_FILE, "w") as f:
            for r in results_sc: f.write(json.dumps(r) + "")
        with open(SC_CHECKPOINT, "w") as f:
            json.dump({"last_index": idx+1}, f)
        acc  = sum(r["correct"] for r in results_sc) / len(results_sc) * 100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}] acc={acc:.1f}%  ({mins:.1f} min)")

with open(SC_RESULTS_FILE, "w") as f:
    for r in results_sc: f.write(json.dumps(r) + "")

n_correct_sc = sum(r["correct"] for r in results_sc)
n_empty_sc   = sum(r["empty"]   for r in results_sc)
sc_acc = n_correct_sc / len(results_sc) * 100

print(f"=== FINAL RESULT (SC) ===")
print(f"  FT 1.5B SC(5):   {n_correct_sc}/{len(results_sc)} = {sc_acc:.1f}%")
print(f"  Empty answers:   {n_empty_sc}")
print(f"  Compute:         7.5B pp (5 voting passes)")

Evaluating 700 questions — FT Self-Consistency (5 votes)...
Compute: 7.5B pp per question (5 voting passes)
------------------------------------------------------------


FT-1.5B-SC(5):   0%|          | 0/700 [00:00<?, ?it/s]

  [ 50] acc=62.0%  (2.2 min)
  [100] acc=65.0%  (4.3 min)
  [150] acc=64.7%  (6.2 min)
  [200] acc=62.0%  (8.2 min)
  [250] acc=63.6%  (10.2 min)
  [300] acc=65.3%  (11.8 min)
  [350] acc=64.6%  (13.9 min)
  [400] acc=64.2%  (16.0 min)
  [450] acc=64.0%  (18.0 min)
  [500] acc=64.0%  (20.2 min)
  [550] acc=63.1%  (22.4 min)
  [600] acc=61.8%  (24.7 min)
  [650] acc=62.2%  (26.7 min)
  [700] acc=61.1%  (29.2 min)
=== FINAL RESULT (SC) ===
  FT 1.5B SC(5):   428/700 = 61.1%
  Empty answers:   55
  Compute:         7.5B pp (5 voting passes)


In [22]:
# CELL 18 — Full evaluation (N=300) — Solo (1 pass)
print(f"Evaluating {CONFIG["eval_n"]} questions — FT Solo (1 forward pass)...")
print(f"Compute: 1.5B pp per question")
print("-"*60)

results_solo = []; start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f: ck = json.load(f)
    start_idx = ck.get("last_index", 0)
    if start_idx >= len(eval_data):
        start_idx = 0
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results_solo = [json.loads(l) for l in f if l.strip()]
    if start_idx:
        print(f"Resumed from {start_idx}")

t0 = time.time()
for idx in tqdm(range(start_idx, len(eval_data)), desc="FT-1.5B-Solo"):
    item = eval_data[idx]
    try:
        raw  = run_ft_solo_race(item)
        pred = normalize_race(extract_race_label(raw))
        gt   = normalize_race(item["answer"])
        results_solo.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"],
            "raw_output": raw, "final_answer": pred,
            "correct": (pred == gt), "empty": (pred == ""),
        })
    except Exception as e:
        results_solo.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"], "raw_output": "",
            "final_answer": "", "correct": False, "empty": True,
            "error": str(e)
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in results_solo: f.write(json.dumps(r) + "")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx+1}, f)
        acc = sum(r["correct"] for r in results_solo) / len(results_solo) * 100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}] acc={acc:.1f}%  ({mins:.1f}min)")

with open(CONFIG["results_file"], "w") as f:
    for r in results_solo: f.write(json.dumps(r) + "")

n_correct = sum(r["correct"] for r in results_solo)
n_empty   = sum(r["empty"]   for r in results_solo)
solo_acc  = n_correct / len(results_solo) * 100
print(f"=== FINAL RESULT (Solo) ===")
print(f"  FT 1.5B Solo:    {n_correct}/{len(results_solo)} = {solo_acc:.1f}%")
print(f"  Empty answers:   {n_empty}")
print(f"  Compute:         1.5B pp (1 forward pass)")

Evaluating 700 questions — FT Solo (1 forward pass)...
Compute: 1.5B pp per question
------------------------------------------------------------


FT-1.5B-Solo:   0%|          | 0/700 [00:00<?, ?it/s]

  [ 50] acc=62.0%  (0.4min)
  [100] acc=61.0%  (0.9min)
  [150] acc=61.3%  (1.3min)
  [200] acc=60.0%  (1.6min)
  [250] acc=61.6%  (2.1min)
  [300] acc=64.0%  (2.4min)
  [350] acc=62.9%  (2.8min)
  [400] acc=62.5%  (3.2min)
  [450] acc=62.2%  (3.6min)
  [500] acc=62.2%  (4.0min)
  [550] acc=61.5%  (4.5min)
  [600] acc=60.5%  (4.9min)
  [650] acc=61.4%  (5.3min)
  [700] acc=61.0%  (5.8min)
=== FINAL RESULT (Solo) ===
  FT 1.5B Solo:    427/700 = 61.0%
  Empty answers:   87
  Compute:         1.5B pp (1 forward pass)


In [24]:
# CELL 20 — Final comparison table
print("="*65)
print("RACE — FULL COMPUTE-ACCURACY COMPARISON")
print("="*65)
print(f"  Condition              | Compute   | Accuracy")
print(f"  -----------------------|-----------|----------")
print(f"  FT 1.5B Solo (this)   | 1.5B pp   | {solo_acc:.1f}%")
print(f"  FT 1.5B SC-5 (this)   | 7.5B pp   | {sc_acc:.1f}%")
print()
gap = sc_acc - solo_acc
print(f"  SC-5 vs Solo: {gap:+.1f} pts")
print()
if gap > 5:
    print(f"  VERDICT: Self-consistency adds {gap:.1f} pts on RACE.")
    print("  Voting improves answer stability on multi-choice tasks.")
elif gap > 0:
    print(f"  VERDICT: SC adds a modest {gap:.1f} pts at 5× the compute.")
    print("  Marginal gain — efficiency may favour the solo approach.")
else:
    print("  VERDICT: SC does not improve over Solo on RACE.")
    print("  Greedy decoding is already near-optimal for this task.")

RACE — FULL COMPUTE-ACCURACY COMPARISON
  Condition              | Compute   | Accuracy
  -----------------------|-----------|----------
  FT 1.5B Solo (this)   | 1.5B pp   | 61.0%
  FT 1.5B SC-5 (this)   | 7.5B pp   | 61.1%

  SC-5 vs Solo: +0.1 pts

  VERDICT: SC adds a modest 0.1 pts at 5× the compute.
  Marginal gain — efficiency may favour the solo approach.
